In [1]:
# import libraries
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime

In [3]:
# define crypto assets
crypto_tickers = {
    "BTC": "BTC-USD",
    "ETH": "ETH-USD",
    "BNB": "BNB-USD",
    "SOL": "SOL-USD",
    "ADA": "ADA-USD",
    "XRP": "XRP-USD",
    "DOT": "DOT-USD",
    "DOGE": "DOGE-USD",
    "LTC": "LTC-USD"
}

In [4]:
# define time range
start_date = "2020-01-01"
end_date = datetime.today().strftime("%Y-%m-%d")

In [5]:
# download the data
all_data = []

for coin, ticker in crypto_tickers.items():
    print(f"Downloading {coin} ({ticker})...")
    
    try:
        df = yf.download(
            ticker,
            start=start_date,
            end=end_date,
            progress=False,
            auto_adjust=False,
            threads=False
        )
        
        if df.empty:
            print(f"Skipped {coin}: empty dataframe")
            continue

        # if yfinance returns multi-index columns, flatten them
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        df = df.reset_index()

        # keep only needed columns
        expected_cols = ["Date", "Open", "High", "Low", "Close", "Volume"]
        df = df[[col for col in expected_cols if col in df.columns]]
         # add coin label
        df["coin"] = coin

        # reorder columns
        df = df[["Date", "coin", "Open", "High", "Low", "Close", "Volume"]]

        all_data.append(df)
        print(f"{coin}: {len(df)} rows")

    except Exception as e:
        print(f"Failed for {coin}: {e}")

crypto_df = pd.concat(all_data, ignore_index=True)

BTC: 2277 rows
ETH: 2277 rows
BNB: 2277 rows
SOL: 2177 rows
ADA: 2277 rows
XRP: 2277 rows
DOT: 2045 rows
DOGE: 2277 rows
LTC: 2277 rows


In [6]:
crypto_df.head()

Price,Date,coin,Open,High,Low,Close,Volume
0,2020-01-01,BTC,7194.892090,7254.330566,7174.944336,7200.174316,18565664997
1,2020-01-02,BTC,7202.551270,7212.155273,6935.270020,6985.470215,20802083465
2,2020-01-03,BTC,6984.428711,7413.715332,6914.996094,7344.884277,28111481032
3,2020-01-04,BTC,7345.375488,7427.385742,7309.514160,7410.656738,18444271275
4,2020-01-05,BTC,7410.451660,7544.497070,7400.535645,7411.317383,19725074095


In [7]:
crypto_df.columns

Index(['Date', 'coin', 'Open', 'High', 'Low', 'Close', 'Volume'], dtype='object', name='Price')

In [8]:
crypto_df.groupby("coin").size()

coin
ADA     2277
BNB     2277
BTC     2277
DOGE    2277
DOT     2045
ETH     2277
LTC     2277
SOL     2177
XRP     2277
dtype: int64

In [9]:
crypto_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20161 entries, 0 to 20160
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Date    20161 non-null  datetime64[ns]
 1   coin    20161 non-null  object        
 2   Open    20161 non-null  float64       
 3   High    20161 non-null  float64       
 4   Low     20161 non-null  float64       
 5   Close   20152 non-null  float64       
 6   Volume  20161 non-null  int64         
dtypes: datetime64[ns](1), float64(4), int64(1), object(1)
memory usage: 1.1+ MB


In [10]:
crypto_df.describe()

Price,Date,Open,High,Low,Close,Volume
count,20161,20161.000000,20161.000000,20161.000000,20152.000000,2.016100e+04
mean,2023-03-01 03:58:03.577203712,5764.950290,5879.756699,5642.510153,5766.932090,7.721087e+09
min,2020-01-01 00:00:00,0.001540,0.001612,0.001247,0.001537,6.520200e+05
25%,2021-08-19 00:00:00,0.613304,0.633577,0.592819,0.613469,4.889751e+08
50%,2023-03-02 00:00:00,37.421608,39.359974,35.455112,37.502449,1.607578e+09
75%,2024-09-12 00:00:00,519.968628,534.177856,506.370361,520.080658,6.671571e+09
max,2026-03-26 00:00:00,124752.140625,126198.070312,123196.046875,124752.531250,3.509679e+11
std,NaN,18500.425464,18834.166194,18141.361637,18503.705561,1.454051e+10


In [11]:
crypto_df.to_csv("../data/raw/crypto_prices_raw.csv", index=False)